In [1]:
import sys
from pathlib import Path
import time
import pandas as pd
import numpy as np

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from src.pipeline import EmailAnalysisPipeline

pipeline = EmailAnalysisPipeline(base_dir=BASE_DIR)


In [2]:
test_emails_dir = BASE_DIR / 'test_emails'
eml_files = list(test_emails_dir.glob('*.eml'))

results_list = []

for eml_file in eml_files:
    try:
        with open(eml_file, 'rb') as f:
            eml_content = f.read()
        
        start_time = time.time()
        analysis_result = pipeline.analyze_email(eml_content, use_parallel=True)
        analysis_time = time.time() - start_time
        
        aggregation = analysis_result.get('aggregation_result', {})
        final_verdict = aggregation.get('final_verdict', 0)
        final_score = aggregation.get('final_score', 0.0)
        detected_language = analysis_result.get('detected_language', 'unknown')
        
        verdict_text = 'PHISHING' if final_verdict == 1 else 'LEGITIMATE'
        
        results_list.append({
            'filename': eml_file.name,
            'analysis_time': analysis_time,
            'language': detected_language,
            'verdict': verdict_text,
            'final_score': final_score
        })
    except Exception as e:
        results_list.append({
            'filename': eml_file.name,
            'analysis_time': None,
            'language': 'error',
            'verdict': 'ERROR',
            'final_score': None
        })

df_results = pd.DataFrame(results_list)


In [3]:
df_valid = df_results[df_results['analysis_time'].notna()]

if len(df_valid) > 0:
    print(f"Всего проанализировано: {len(df_valid)} писем")
    print(f"\nОбщая статистика времени анализа:")
    print(f"Среднее: {df_valid['analysis_time'].mean():.3f} с")
    print(f"Медиана: {df_valid['analysis_time'].median():.3f} с")
    print(f"Мин: {df_valid['analysis_time'].min():.3f} с")
    print(f"Макс: {df_valid['analysis_time'].max():.3f} с")
    print(f"Ст. откл.: {df_valid['analysis_time'].std():.3f} с")
    
    print("\nРаспределение по языкам:")
    for lang, count in df_valid['language'].value_counts().items():
        lang_name = "Русский" if lang == 'ru' else "Английский" if lang == 'en' else lang
        print(f"{lang_name}: {count} писем")


Всего проанализировано: 50 писем

Общая статистика времени анализа:
Среднее: 2.330 с
Медиана: 1.135 с
Мин: 0.046 с
Макс: 30.594 с
Ст. откл.: 4.957 с

Распределение по языкам:
Русский: 42 писем
Английский: 8 писем


In [6]:
if len(df_valid) > 0:
    languages = df_valid['language'].unique()
    stats_by_language = []
    
    for lang in sorted(languages):
        df_lang = df_valid[df_valid['language'] == lang]
        
        if len(df_lang) > 0:
            lang_name = "Русский" if lang == 'ru' else "Английский" if lang == 'en' else lang
            
            stats = {
                'Язык': lang_name,
                'Кол-во писем': len(df_lang),
                'Среднее время (с)': df_lang['analysis_time'].mean(),
                'Медиана (с)': df_lang['analysis_time'].median(),
                'Мин (с)': df_lang['analysis_time'].min(),
                'Макс (с)': df_lang['analysis_time'].max(),
                'Ст. откл. (с)': df_lang['analysis_time'].std()
            }
            stats_by_language.append(stats)
            
            print(f"{lang_name}:")
            print(f"  Количество: {stats['Кол-во писем']}")
            print(f"  Среднее время: {stats['Среднее время (с)']:.3f} с")
            print(f"  Медиана: {stats['Медиана (с)']:.3f} с")
            print(f"  Мин: {stats['Мин (с)']:.3f} с")
            print(f"  Макс: {stats['Макс (с)']:.3f} с")
            print(f"  Ст. откл.: {stats['Ст. откл. (с)']:.3f} с\n")
    
    df_stats = pd.DataFrame(stats_by_language)
    print("Сводная таблица:")
    print(df_stats.to_string(index=False))


Английский:
  Количество: 8
  Среднее время: 0.298 с
  Медиана: 0.086 с
  Мин: 0.046 с
  Макс: 1.791 с
  Ст. откл.: 0.604 с

Русский:
  Количество: 42
  Среднее время: 2.717 с
  Медиана: 1.200 с
  Мин: 0.318 с
  Макс: 30.594 с
  Ст. откл.: 5.324 с

Сводная таблица:
      Язык  Кол-во писем  Среднее время (с)  Медиана (с)  Мин (с)  Макс (с)  Ст. откл. (с)
Английский             8           0.298133     0.085649 0.046052  1.790684       0.603833
   Русский            42           2.717165     1.200341 0.318456 30.593734       5.323545
